In [2]:
# Coffee Shop Profit & Loss Analysis (Dataset-specific)
# Google Colab friendly script.

!pip install -q matplotlib pandas numpy scikit-learn plotly python-pptx openpyxl

import io
import os
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from pptx import Presentation
from pptx.util import Inches
from google.colab import files

# === Upload file ===
print("Please upload your dataset file (Coffee Shop Sales CSV).")
uploaded = files.upload()
if len(uploaded) == 0:
    raise SystemExit("No file uploaded. Rerun and upload dataset.")

fname = list(uploaded.keys())[0]
print(f"Uploaded: {fname}")
df = pd.read_csv(io.BytesIO(uploaded[fname]))

# Normalize column names
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

# === Map dataset-specific columns ===
cols_map = {
    'date': 'transaction_date',
    'product': 'product_category',  # Adjusted for your dataset
    'quantity': 'transaction_qty',
    'unit_price': 'unit_price',
    'revenue': None,
    'cost': None
}

# Ensure types
df[cols_map['date']] = pd.to_datetime(df[cols_map['date']], errors='coerce')
df[cols_map['quantity']] = pd.to_numeric(df[cols_map['quantity']], errors='coerce')
df[cols_map['unit_price']] = pd.to_numeric(df[cols_map['unit_price']], errors='coerce')

# === Compute revenue, cost, profit ===
df['revenue'] = df[cols_map['unit_price']] * df[cols_map['quantity']]
cols_map['revenue'] = 'revenue'

# Assume cost = 50% of unit price × quantity
df['cost'] = (df[cols_map['unit_price']] * 0.5) * df[cols_map['quantity']]
cols_map['cost'] = 'cost'

# Profit
df['profit'] = df['revenue'] - df['cost']

# === Save cleaned dataset ===
df.to_csv('cleaned_dataset.csv', index=False)
print("Cleaned dataset saved to cleaned_dataset.csv")

# === Aggregate per product ===
agg = df.groupby(cols_map['product']).agg(
    total_profit=('profit', 'sum'),
    total_revenue=('revenue', 'sum'),
    quantity=(cols_map['quantity'], 'sum')
).reset_index().sort_values('total_profit', ascending=False)

agg.to_csv('product_profit_summary.csv', index=False)
print("Saved product-level profit summary.")

# === Visualizations ===
fig = px.bar(agg.head(15), x=cols_map['product'], y='total_profit', title='Top 15 Products by Profit')
fig.show()

fig2 = px.pie(agg, names=cols_map['product'], values='total_profit', title='Profit Share by Product')
fig2.show()

if cols_map['date']:
    ts = df.set_index(cols_map['date']).resample('M').agg(monthly_profit=('profit','sum')).reset_index()
    fig3 = px.line(ts, x=cols_map['date'], y='monthly_profit', title='Monthly Profit Over Time')
    fig3.show()

# === Predictive Modeling ===
X = df[[cols_map['quantity'], cols_map['unit_price']]]
y = df['profit']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
preds = model.predict(X_test)

print("Model Performance:")
print("MAE:", mean_absolute_error(y_test, preds))
print("R2:", r2_score(y_test, preds))

# === Recommendations ===
recommendations = []
top = agg.head(5)
low = agg.tail(5).sort_values('total_profit')

recommendations.append("Top profitable products:")
for _, r in top.iterrows():
    recommendations.append(f" - {r[cols_map['product']]}: Profit {r['total_profit']:.2f}")

recommendations.append("\nLoss-making products:")
for _, r in low.iterrows():
    recommendations.append(f" - {r[cols_map['product']]}: Profit {r['total_profit']:.2f}")

recommendations.append("\nStrategies:")
recommendations.append(" - Increase prices slightly for top sellers.")
recommendations.append(" - Negotiate supplier costs for low-margin items.")
recommendations.append(" - Promote bundles of high-margin with popular items.")

with open('recommendations.txt','w') as f:
    f.write('\n'.join(recommendations))
print("Recommendations saved.")

# === PowerPoint Summary ===
prs = Presentation()
slide = prs.slides.add_slide(prs.slide_layouts[0])
slide.shapes.title.text = 'Profit & Loss Analysis - Coffee Shop'
slide.placeholders[1].text = f'Generated {datetime.now().strftime("%Y-%m-%d %H:%M")}'

slide = prs.slides.add_slide(prs.slide_layouts[1])
slide.shapes.title.text = 'Key Metrics'
txt = slide.placeholders[1].text_frame
txt.text = f"Rows: {df.shape[0]}\nColumns: {df.shape[1]}\nTop Product: {agg.iloc[0][cols_map['product']]}"

prs.save('analysis_summary.pptx')
print("PowerPoint saved.")

print("\nDone ✅ Check generated files: cleaned_dataset.csv, product_profit_summary.csv, recommendations.txt, analysis_summary.pptx")


Please upload your dataset file (Coffee Shop Sales CSV).


Saving Coffee Shop Sales (1) - Transactions.csv to Coffee Shop Sales (1) - Transactions (1).csv
Uploaded: Coffee Shop Sales (1) - Transactions (1).csv
Cleaned dataset saved to cleaned_dataset.csv
Saved product-level profit summary.


/tmp/ipython-input-1442461987.py:84: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



Model Performance:
MAE: 1.0729613736035832e-05
R2: 0.9999992371772914
Recommendations saved.
PowerPoint saved.

Done ✅ Check generated files: cleaned_dataset.csv, product_profit_summary.csv, recommendations.txt, analysis_summary.pptx
